## ES-VAE with Squared Geodesic Loss (Classification)

In [14]:
import numpy as np
import pandas as pd
import random
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, TensorDataset

from val_test import *
from print_results import *
from functionsgpu_fast import *

import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)
dtype = torch.float32

if device.type == "cuda":
    idx = device.index if device.index is not None else torch.cuda.current_device()
    print(torch.cuda.get_device_name(idx))

SEED = 42
def deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

deterministic(SEED)
# Enable (as much as possible) deterministic operations
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

tslen = 200

cuda:1
NVIDIA RTX A5000


## Load y_lesion and Participant IDs

In [3]:
# Load y_lesion and participant IDs
# Participant ID for each row of dataset (same order as files from csv_r)
participant_ids = np.loadtxt('/mnt/sdb/arafat/stroke_riemann/labels_data/pids.txt')
demo_df = pd.read_csv('/mnt/sdb/arafat/stroke_riemann/labels_data/demo_data.csv')
id_to_lesion = dict(zip(demo_df['s'].astype(int), demo_df['LesionLeft']))
y_lesion = np.array([id_to_lesion[int(pid)] for pid in participant_ids])

print("y_lesion shape:", y_lesion.shape)
print("First 10 participant_ids:", participant_ids[:10])

y_lesion shape: (155,)
First 10 participant_ids: [ 1.  2.  3.  4.  5.  6.  7.  8. 10. 11.]


## Data Loading

In [4]:
def loading(filename, tslen):
    with open('{}/betas_aligned{}.pkl'.format(filename, tslen), 'rb') as f:
        betas_aligned = pickle.load(f)
    with open('{}/mu{}.pkl'.format(filename, tslen), 'rb') as f:
        mu = pickle.load(f)
    with open('{}/tangent_vecs{}.pkl'.format(filename, tslen), 'rb') as f:
        tangent_vec_all = pickle.load(f)
    return betas_aligned, mu, tangent_vec_all

betas_aligned_all, mu_all_t, tangent_vec_all = loading('/mnt/sdb/arafat/stroke_riemann/aligned_data',tslen)
mu_all_t_tensor = torch.from_numpy(mu_all_t).to(device=device, dtype=torch.float32)
betas_aligned = np.array(betas_aligned_all)
betas_aligned = betas_aligned.transpose(1, 2, 3, 0)
print(betas_aligned.shape, tangent_vec_all.shape, mu_all_t.shape)

(32, 3, 200, 155) (32, 3, 200, 155) (32, 3, 200)


In [5]:
K = 32
M = 3
T = tslen
nsamples = 155

tangent_flat = tangent_vec_all.reshape((K*M*T, nsamples))
print(tangent_flat.shape)

(19200, 155)


## Nonlinear Tangent VAE

In [8]:
class NonlinearVAE(nn.Module):
    """NonlinearVAE"""
    def __init__(self, D, R, H=128, dropout=0.25):
        super().__init__()
        # Encoder layers
        self.W1 = nn.Linear(D, H, bias=False)        # input -> hidden
        self.W2_mu = nn.Linear(H, R, bias=False)     # hidden -> latent mean
        self.W2_logvar = nn.Linear(H, R)             # hidden -> latent logvar
        self.dropout = nn.Dropout(p=dropout)
        
        # Decoder layers
        self.dec1 = nn.Linear(R, 32, bias=False)
        self.dec2 = nn.Linear(32, D, bias=False)

    def encode(self, x):
        h = torch.tanh(self.W1(x))
        h = self.dropout(h)
        mu = self.W2_mu(h)
        logvar = self.W2_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h_recon = torch.tanh(self.dec1(z))
        x_hat = self.dec2(h_recon)
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z

In [9]:
class ESVAE(nn.Module):
    """rVAE: encode tangent vectors, decode to tangent, but (during
    training) compare reconstructions on the manifold via an exp map.

    - Inputs: tangent vectors (N, D) as in the VAE section.
    - Decoder: produces tangent vectors.
    - Training loss: uses expmap(mu, v_hat) vs. original manifold trajectory.
    """
    def __init__(self, base_vae, mu_shape, expmap):
        super().__init__()
        self.vae = base_vae
        # mean shape, used for exponential map when mapping back to manifold
        self.register_buffer("mu_shape", mu_shape)
        self.expmap = expmap

    def forward(self, x):
        """Forward on tangent vectors.

        x : (N, D) tangent vectors
        Returns
        -------
        x_man_hat : (N, D) manifold trajectory flattened (via expmap)
        mu_z, logvar, z, v_hat : usual VAE outputs (in tangent space)
        """
        v_hat, mu_z, logvar, z = self.vae(x)   # tangent reconstruction

        # Map reconstructed tangent field back to the manifold
        B = v_hat.shape[0]
        v_hat_reshaped = v_hat.view(B, K, M, T)
        mu = self.mu_shape.view(K, M, T)
        x_recon_man = self.expmap(mu, v_hat_reshaped)   # (B, K, M, T)
        x_recon_man = x_recon_man.view(B, -1)

        return x_recon_man, mu_z, logvar, z, v_hat

def esvae_loss(x_man, x_hat_man, mu, logvar, beta=1e-4):
    dist = squared_geodesic_distance(x_man, x_hat_man, K, M, T)
    recon = dist.mean()
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    avg_kl = kl.mean()
    return recon + beta * avg_kl, recon, avg_kl

## Getting Orginal Trajectories on Manifold

In [10]:
betas = np.array(betas_aligned_all)
print(betas.shape)   # (155, 32, 3, 200)

N, K, M, T = betas.shape   # correct order

# Shape mean as before (kept for potential manifold utilities)
mu_shape = torch.from_numpy(
    mu_all_t.reshape(-1).astype(np.float32)
).to(device)

print("mu_shape:", mu_shape.shape)      # should be (19200,)

(155, 32, 3, 200)
mu_shape: torch.Size([19200])


## Training Function for Each Fold (ES-VAE)

In [16]:
def train_esvae_fold(X_tan_train, X_man_train, R=10, num_epochs=2000, batch_size=145,
                            lr=1e-3, betakl=2**(-3), seed=42):
    """Train a fresh KendallRVAE on a training subset.

    X_tan_train : (N_train, D) tangent vectors
    X_man_train : (N_train, K*M*T) manifold trajectories (flattened)
    """
    deterministic(seed)
    D = X_tan_train.shape[1]

    dataset = TensorDataset(X_tan_train, X_man_train)
    g = torch.Generator(device=device).manual_seed(seed)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False, generator=g)

    base_vae_fold = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    model_fold = ESVAE(base_vae_fold, mu_shape, exp_gpu_batch).to(device=device, dtype=dtype)
    opt_fold = torch.optim.Adam(model_fold.parameters(), lr=lr)

    model_fold.train()
    for epoch in range(num_epochs):
        epoch_loss, epoch_recon, epoch_kl, num_samples = 0.0, 0.0, 0.0, 0

        for (x_tan_batch, x_man_batch) in loader:
            x_tan_batch = x_tan_batch.to(device=device, dtype=dtype)
            x_man_batch = x_man_batch.to(device=device, dtype=dtype)

            opt_fold.zero_grad(set_to_none=True)
            x_hat_man, mu, logvar, z, v_hat = model_fold(x_tan_batch)
            loss_train, recon_train, kl_train = esvae_loss(x_man_batch, x_hat_man, mu, logvar, beta=betakl)
            loss_train.backward()
            opt_fold.step()

            bs = x_tan_batch.size(0)
            epoch_loss += loss_train.item() * bs
            epoch_recon += recon_train.item() * bs
            epoch_kl += kl_train.item() * bs
            num_samples += bs

    model_fold.eval()
    return model_fold

## ES-VAE Cross Validation

In [17]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# classification models
models = {'KNN': KNeighborsClassifier()}

# Leave-10-participants-out CV: 5 validation + 5 test (disjoint). Two rounds so every subject is validated and tested once.
n = len(y_lesion)
all_results_nested = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models.keys()}
participant_ids = np.asarray(participant_ids)
all_results_validation = {name: {'targets': [], 'preds': []} for name in models.keys()}

n_folds = 30
R = 38
X_tan = torch.from_numpy(tangent_flat.T.astype(np.float32)).to(device=device, dtype=dtype)
X_man = torch.from_numpy(betas.reshape(betas.shape[0], -1).astype(np.float32)).to(device=device, dtype=dtype)

for k in tqdm(range(n_folds), total=n_folds, desc='KT-RSV folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids[j] in train_pids])
    test_idx = np.array([j for j in range(n) if participant_ids[j] in test_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids[j] in validation_pids])

    if len(train_idx) == 0 or len(test_idx) == 0 or len(validation_idx) == 0:
        continue

    fold_seed = SEED + k
    deterministic(fold_seed)

    # Slice tangent and manifold data for this fold
    X_tan_train = X_tan[train_idx]
    X_man_train = X_man[train_idx]

    X_tan_val = X_tan[validation_idx]
    X_man_val = X_man[validation_idx]

    bsize = X_tan_train.shape[0]

    # Train fold-specific ESVAE on train participants only
    model_fold = train_esvae_fold(X_tan_train, X_man_train, R=R, batch_size=bsize,
                                         num_epochs=25, lr=1e-5, seed=fold_seed)

    # Extract latent means (mu) for train, validation, and test using the fold-specific encoder
    with torch.no_grad():
        mu_train_fold, _ = model_fold.vae.encode(X_tan_train)
        mu_validation_fold, _ = model_fold.vae.encode(X_tan[validation_idx])
        mu_test_fold, _ = model_fold.vae.encode(X_tan[test_idx])

    Z_train_fold = mu_train_fold.detach().cpu().numpy()
    Z_validation_fold = mu_validation_fold.detach().cpu().numpy()
    Z_test_fold = mu_test_fold.detach().cpu().numpy()

    y_train_fold = y_lesion[train_idx]
    y_validation_fold = y_lesion[validation_idx]
    y_test_fold = y_lesion[test_idx]

    # Train and evaluate each regressor on this fold's latents
    for name, model_reg in models.items():
        # fresh instance with same random_state
        m = type(model_reg)(**model_reg.get_params())
        m.fit(Z_train_fold, y_train_fold)

        train_preds = m.predict(Z_train_fold)

        # Validation predictions
        validation_preds = m.predict(Z_validation_fold)
        
        # Test predictions
        test_preds = m.predict(Z_test_fold)

        # Store validation results for per-fold reporting
        all_results_validation[name]['targets'].extend(y_validation_fold.tolist())
        all_results_validation[name]['preds'].extend(validation_preds.tolist())

        # Store test results for global metrics
        all_results_nested[name]['targets'].extend(y_test_fold.tolist())
        all_results_nested[name]['preds'].extend(test_preds.tolist())
        all_results_nested[name]['subjects'].extend(participant_ids[test_idx].tolist())

        # Per-fold train metrics
        f1_train = f1_score(y_train_fold, train_preds, average='macro')

        # Per-fold validation metrics
        f1_val = f1_score(y_validation_fold, validation_preds, average='macro')

        # Per-fold test metrics
        f1_test = f1_score(y_test_fold, test_preds, average='macro')
        
        print(f"Fold {k + 1:02d} | Train: F1={f1_train:.2f}, Validation: F1={f1_val:.2f},",
              f"Test: F1={f1_test:.2f}")

results_nested_df = print_results_clf(all_results_validation, all_results_nested, models)
results_nested_df

KT-RSV folds:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 01 | Train: F1=0.89, Validation: F1=0.30, Test: F1=0.44
Fold 02 | Train: F1=0.85, Validation: F1=0.60, Test: F1=0.60
Fold 03 | Train: F1=0.83, Validation: F1=0.56, Test: F1=0.44
Fold 04 | Train: F1=0.79, Validation: F1=0.30, Test: F1=0.44
Fold 05 | Train: F1=0.78, Validation: F1=0.82, Test: F1=1.00
Fold 06 | Train: F1=0.87, Validation: F1=1.00, Test: F1=1.00
Fold 07 | Train: F1=0.88, Validation: F1=1.00, Test: F1=1.00
Fold 08 | Train: F1=0.86, Validation: F1=1.00, Test: F1=1.00
Fold 09 | Train: F1=0.78, Validation: F1=1.00, Test: F1=1.00
Fold 10 | Train: F1=0.85, Validation: F1=0.44, Test: F1=1.00
Fold 11 | Train: F1=0.85, Validation: F1=1.00, Test: F1=1.00
Fold 12 | Train: F1=0.85, Validation: F1=1.00, Test: F1=1.00
Fold 13 | Train: F1=0.82, Validation: F1=1.00, Test: F1=1.00
Fold 14 | Train: F1=0.87, Validation: F1=1.00, Test: F1=1.00
Fold 15 | Train: F1=0.83, Validation: F1=1.00, Test: F1=1.00
Fold 16 | Train: F1=0.86, Validation: F1=0.25, Test: F1=0.30
Fold 17 | Train: F1=0.86

,Accuracy,F1 (macro),Precision (macro),Recall (macro)
KNN,0.916129,0.82539,0.856002,0.804762


In [18]:
from ci_class import *

ci_results = {}

name = "KNN"

ci_results[name] = subject_bootstrap_ci_class(
    all_results_nested[name]['targets'],
    all_results_nested[name]['preds'],
    all_results_nested[name]['subjects'])

pd.DataFrame(ci_results['KNN'])

,Accuracy,F1 (weighted),F1 (macro),Precision (weighted),Precision (macro),Recall (weighted),Recall (macro)
mean,0.916129,0.912133,0.82539,0.914881,0.856002,0.916129,0.804762
ci,"[0.885, 0.949]","[0.878, 0.948]","[0.754, 0.897]","[0.883, 0.952]","[0.79, 0.931]","[0.885, 0.949]","[0.728, 0.886]"


In [19]:
from sklearn.metrics import classification_report

# Classification report for each model
target_names = ['LesionRight (0)', 'LesionLeft (1)', 'Healthy (2)']
d = all_results_nested[name]
print(f"\n=== {name} ===")
print(classification_report(d['targets'], d['preds'], target_names=target_names, zero_division=0))


=== KNN ===
                 precision    recall  f1-score   support

LesionRight (0)       0.91      0.70      0.79        30
 LesionLeft (1)       0.71      0.71      0.71        14
    Healthy (2)       0.94      1.00      0.97       111

       accuracy                           0.92       155
      macro avg       0.86      0.80      0.83       155
   weighted avg       0.91      0.92      0.91       155

